# v3 rebuild — Stage 3 · modality ablations + two-tower fusion

Answers Reviewer 3.3 / Reviewer 4.5 ("justify that fusion actually helps") under the
**same** subject-disjoint LOSO protocol as Stage 2, on the Stage-1 clean features:

* **EEG-only** (96 channel features), **physio-only** (ECG+EDA+EMG+RESP, 32),
  **fused** (128) for the four leading models (XGBoost, HistGB, RF, MLP);
* an explicit **two-tower neural fusion** (separate EEG / physio towers) with the
  corrected training loop (per-fold training-median NaN imputation, stratified
  validation, class-weighted loss — see the markdown in `12_v3_twotower_fix.ipynb`
  for the collapse it fixes);
* fusion-benefit table: fused vs the best single modality per model, paired Wilcoxon;
* a within-subject XGBoost upper bound by modality.

Writes under `v3_rebuild/ablation/`: `oof_{modality}_{model}.npz`,
`oof_fused_two_tower.npz`, `modality_ablation.csv`, `fusion_benefit.csv`,
`within_subject_ablation_xgb.csv`.

> **Runtime: ~1–3 h** (12 sklearn runs × 52 folds + the two-tower). Re-running
> supersedes the two-tower row computed by `12_v3_twotower_fix.ipynb`; both write the
> same corrected result (two-tower ≈0.406).


In [ ]:
import copy, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split


def _root(start=Path.cwd()):
    for d in [start, *start.parents]:
        if (d / "src" / "rebuild" / "models_registry.py").exists():
            return d
    raise SystemExit("run notebook from the module root or src/")
REPO = _root()
sys.path.insert(0, str(REPO / "src" / "rebuild"))

from ablations import load, loso_sklearn, loso_twotower, indices_of, ABLATION_MODELS
from models_registry import make_models

OUT_ROOT = REPO / "output" / "research_outputs" / "fusion_training" / "v3_rebuild"
ABL = OUT_ROOT / "ablation"
ABL.mkdir(parents=True, exist_ok=True)
SEED = 42
print("out:", ABL)

In [ ]:
df, X, y, subjects, feats = load()
n_class = int(y.max()) + 1
registry = make_models(seed=SEED)
print("rows", len(df), "| features", X.shape[1], "| subjects",
      len(np.unique(subjects)), "| classes", n_class)
print("ablation models:", ABLATION_MODELS)

### Modality × model LOSO (EEG-only / physio-only / fused)

In [ ]:
rows = []
for subset in ["eeg", "physio", "fused"]:
    idx = indices_of(feats, X, subset)
    for mn in ABLATION_MODELS:
        mf, yt, yp = loso_sklearn(lambda: copy.deepcopy(registry[mn]),
                                  X[:, idx], y, subjects, n_class)
        rows.append({"modality": subset, "model": mn,
                     "macro_f1_mean": float(mf.mean()), "macro_f1_std": float(mf.std(ddof=1)),
                     "pooled_macro_f1": float(f1_score(yt, yp, average="macro", zero_division=0))})
        print(f"[{subset:>6}][{mn:>22}] macro-F1 {mf.mean():.4f}±{mf.std(ddof=1):.4f}",
              flush=True)
        np.savez_compressed(ABL / f"oof_{subset}_{mn}.npz", y_true=yt, y_pred=yp, folds=mf)
ab = pd.DataFrame(rows)
ab.to_csv(ABL / "modality_ablation.csv", index=False)
ab

### Two-tower neural fusion (corrected training loop)

Same implementation as `12_v3_twotower_fix.ipynb` (imported from `ablations.py`):
missing values are imputed with the training-fold column medians **before** torch
standardization (torch mean/std are not NaN-aware — one NaN used to poison a whole
fold into predicting class 0), the validation slice is stratified, the early-stop
score is fixed-label macro-F1, and the loss is class-weighted.


In [ ]:
t0 = time.time()
mf_tt, yt_tt, yp_tt = loso_twotower(X, y, subjects, feats)
print(f"[fused ][two_tower] macro-F1 {mf_tt.mean():.4f}±{mf_tt.std(ddof=1):.4f}  "
      f"({(time.time()-t0)/60:.1f} min)", flush=True)
np.savez_compressed(ABL / "oof_fused_two_tower.npz",
                    y_true=yt_tt, y_pred=yp_tt, folds=mf_tt)

ab = pd.read_csv(ABL / "modality_ablation.csv")
ab.loc[len(ab)] = ["fused", "two_tower", float(mf_tt.mean()), float(mf_tt.std(ddof=1)),
                   float(f1_score(yt_tt, yp_tt, average="macro", zero_division=0))]
ab.to_csv(ABL / "modality_ablation.csv", index=False)
print("appended two_tower to modality_ablation.csv")

### Fusion benefit: fused vs best single modality (paired Wilcoxon)

In [ ]:
benefit = []
for mn in ABLATION_MODELS:
    f_fus = np.load(ABL / f"oof_fused_{mn}.npz")["folds"]
    best_sub, best_f = None, -np.inf
    for sub in ["eeg", "physio"]:
        f = np.load(ABL / f"oof_{sub}_{mn}.npz")["folds"]
        if f.mean() > best_f:
            best_f, best_sub = f.mean(), sub
    f_best = np.load(ABL / f"oof_{best_sub}_{mn}.npz")["folds"]
    d = f_fus - f_best
    try:
        w, p = stats.wilcoxon(d, zero_method="wilcox")
    except ValueError:
        w, p = float("nan"), float("nan")
    benefit.append({"model": mn, "best_single": best_sub,
                    "fused_mf": float(f_fus.mean()), "single_mf": float(f_best.mean()),
                    "delta": float(d.mean()), "wilcoxon_p": float(p)})
# two-tower vs fused-concat best (fused XGBoost)
f_tt = np.load(ABL / "oof_fused_two_tower.npz")["folds"]
f_xgb = np.load(ABL / "oof_fused_xgboost.npz")["folds"]
d = f_tt - f_xgb
try:
    p_tt = stats.wilcoxon(d, zero_method="wilcox").pvalue
except ValueError:
    p_tt = float("nan")
benefit.append({"model": "two_tower_vs_xgboost_fused", "best_single": "xgboost_fused",
                "fused_mf": float(f_tt.mean()), "single_mf": float(f_xgb.mean()),
                "delta": float(d.mean()), "wilcoxon_p": float(p_tt)})
fb = pd.DataFrame(benefit)
fb.to_csv(ABL / "fusion_benefit.csv", index=False)
fb

### Within-subject XGBoost upper bound by modality (diagnostic)

In [ ]:
ub = []
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
for subset in ["eeg", "physio", "fused"]:
    idx = indices_of(feats, X, subset)
    est = copy.deepcopy(registry["xgboost"])
    est.fit(Xtr[:, idx], ytr)
    ub.append({"modality": subset,
               "within_macro_f1": float(f1_score(yte, est.predict(Xte[:, idx]),
                                                 average="macro", zero_division=0))})
pd.DataFrame(ub).to_csv(ABL / "within_subject_ablation_xgb.csv", index=False)
pd.DataFrame(ub)

### Verdict vs the sealed run

In [ ]:
fb = pd.read_csv(ABL / "fusion_benefit.csv")
ab = pd.read_csv(ABL / "modality_ablation.csv")
tt = ab.loc[ab.model == "two_tower", "macro_f1_mean"].iloc[0]
xf = ab.loc[(ab.model == "xgboost") & (ab.modality == "fused"), "macro_f1_mean"].iloc[0]
print(f"Two-Tower {tt:.4f} vs fused XGBoost {xf:.4f}  ->  delta {tt - xf:+.4f}")
print("Expected (sealed): two-tower 0.4058, fused XGBoost 0.4620, delta -0.056 (p=0.001);",
      "XGBoost +0.072 (p<1e-4), HistGB +0.050 (p=0.002), RF +0.047 (p=0.015),",
      "MLP +0.037 (p=0.105, n.s.).")
fb